In [11]:
%pip install google-cloud-bigquery
%pip install azure-storage-blob
%pip install db-dtypes

%pip install google-cloud-bigquery-storage
%pip install pandas-gbq
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pandas-gbq]5 [pandas-gbq]uthlib]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [25]:
# =========================
# Inicialização Databricks
# =========================

try:
    # Cria widgets no Databricks
    dbutils.library.restartPython()
except NameError:
    # Fallback para execução fora do Databricks (ex: testes locais em Jupyter/VSCode)
    print("⚠️ dbutils não encontrado, usando valores locais para teste.")

⚠️ dbutils não encontrado, usando valores locais para teste.


In [27]:
# =========================
# Imports
# =========================

from pathlib import Path
import os
import json
import tempfile
import datetime as dt

import db_dtypes

# =========================
# 1. Definir raiz do projeto
# =========================

BASE_DIR = Path.cwd()

while not (BASE_DIR / "src").exists():

    if BASE_DIR.parent == BASE_DIR:
        raise FileNotFoundError(
            "❌ Pasta 'src' não encontrada em nenhum nível acima."
        )

    BASE_DIR = BASE_DIR.parent

print(f"✅ BASE_DIR localizado: {BASE_DIR}")

# =========================
# 2. Configuração das credenciais GCP
# =========================

cred_path = None

try:

    # ==========================================
    # Execução Databricks + Azure Key Vault
    # ==========================================

    secret_json = dbutils.secrets.get(
        scope="kvfiaptechprod",
        key="GOOGLE-APPLICATION-CREDENTIALS-JSON"
    )

    # Cria arquivo temporário para a biblioteca
    # google-cloud-bigquery utilizar
    temp_file = tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".json",
        delete=False
    )

    temp_file.write(secret_json)
    temp_file.close()

    cred_path = temp_file.name

    print(
        "✅ Credenciais carregadas do Azure Key Vault."
    )

except NameError:

    # ==========================================
    # Execução Local (VSCode / Jupyter)
    # ==========================================

    print(
        "⚠️ dbutils não encontrado. "
        "Utilizando credenciais locais."
    )

    cred_file = (
        "tough-medley-505300-k1-164371097431.json"
    )

    cred_path = (
        BASE_DIR
        / "credenciais"
        / cred_file
    )

    if not cred_path.exists():

        raise FileNotFoundError(
            f"❌ Arquivo não encontrado: {cred_path}"
        )

    cred_path = str(cred_path)

    print(
        f"✅ Arquivo de credenciais localizado em: "
        f"{cred_path}"
    )

except Exception as e:

    raise RuntimeError(
        f"❌ Erro ao carregar credenciais do Key Vault: {e}"
    )

# =========================
# 3. Variável de ambiente
# =========================

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(
    cred_path
)

print(
    f"✅ GOOGLE_APPLICATION_CREDENTIALS configurado."
)

# =========================
# 4. Validação do arquivo
# =========================

if os.path.exists(str(cred_path)):

    print(
        f"✅ Arquivo de credenciais disponível em: "
        f"{cred_path}"
    )

else:

    raise FileNotFoundError(
        f"❌ Arquivo de credenciais não encontrado: "
        f"{cred_path}"
    )

✅ BASE_DIR localizado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23
⚠️ dbutils não encontrado. Utilizando credenciais locais.
✅ Arquivo de credenciais localizado em: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/credenciais/tough-medley-505300-k1-164371097431.json
✅ GOOGLE_APPLICATION_CREDENTIALS configurado.
✅ Arquivo de credenciais disponível em: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/credenciais/tough-medley-505300-k1-164371097431.json


In [28]:
# =========================
# Parâmetros vindos do Databricks
# =========================

try:

    dbutils.widgets.text("BRONZE_CONTAINER", "bronze")
    dbutils.widgets.text("TABLES", "")

    BRONZE_CONTAINER = (
        dbutils.widgets.get("BRONZE_CONTAINER")
        or "bronze"
    )

    TABLES = (
        dbutils.widgets.get("TABLES").split(",")
        if dbutils.widgets.get("TABLES")
        else []
    )

    print("✅ Widgets carregados.")

except NameError:

    print(
        "⚠️ dbutils não encontrado, usando configuração local."
    )

    BRONZE_CONTAINER = "bronze"

    TABLES = []

print(f"✅ Container Bronze: {BRONZE_CONTAINER}")

⚠️ dbutils não encontrado, usando configuração local.
✅ Container Bronze: bronze


In [29]:
# =========================
# Configuração dos clientes
# =========================

from google.cloud import bigquery
from azure.storage.blob import BlobServiceClient

# =========================
# Cliente BigQuery
# =========================

try:

    if not cred_path:
        raise ValueError(
            "Credenciais GCP não configuradas."
        )

    print(f"✅ Credenciais encontradas em: {cred_path}")

    project_id = os.getenv(
        "GCP_PROJECT_ID",
        "tough-medley-505300-k1"
    )

    bq_client = bigquery.Client.from_service_account_json(
        cred_path,
        project=project_id
    )

    print("✅ Cliente BigQuery inicializado com sucesso.")

except Exception as e:

    print(
        f"❌ Erro ao inicializar cliente BigQuery: {e}"
    )

    bq_client = None

# =========================
# Cliente Azure Blob
# =========================

try:

    storage_account = os.getenv("AZURE_STORAGE_ACCOUNT")
    storage_key = os.getenv("AZURE_STORAGE_KEY")

    blob_service_client = BlobServiceClient(
        account_url=f"https://{storage_account}.blob.core.windows.net",
        credential=storage_key
    )

    print("✅ Cliente Azure Blob inicializado com sucesso.")

except Exception as e:

    print(
        f"❌ Erro ao inicializar cliente Azure Blob: {e}"
    )

    blob_service_client = None

✅ Credenciais encontradas em: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/credenciais/tough-medley-505300-k1-164371097431.json
✅ Cliente BigQuery inicializado com sucesso.
✅ Cliente Azure Blob inicializado com sucesso.


In [31]:
# =========================
# Função de exportação
# =========================

def export_bigquery_table_to_blob(
    source_table: str,
    blob_container: str,
    blob_name: str
):
    """
    Exporta uma tabela do BigQuery
    para Azure Blob Storage em Parquet.
    """

    try:

        query = f"""
        SELECT *
        FROM `{source_table}`
        """

        query_job = bq_client.query(
            query,
            location="US"
        )

        df = query_job.to_dataframe()

        # Auditoria

        df["_ingested_at"] = (
            dt.datetime.now(
                dt.timezone.utc
            ).isoformat()
        )

        df["_source_table"] = source_table

        # Diretório temporário

        temp_dir = BASE_DIR / "tmp"

        temp_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        parquet_file = (
            temp_dir / "temp.parquet"
        )

        df.to_parquet(
            parquet_file,
            index=False
        )

        blob_client = (
            blob_service_client.get_blob_client(
                container=blob_container,
                blob=blob_name
            )
        )

        with open(parquet_file, "rb") as data:

            blob_client.upload_blob(
                data,
                overwrite=True
            )

        print(
            f"✅ Exportado {source_table} "
            f"para azure://{blob_container}/{blob_name}"
        )

    except Exception as e:

        print(
            f"❌ Erro ao exportar "
            f"{source_table}: {e}"
        )

In [33]:
# =========================
# Ingestão Batch
# =========================

import datetime as dt

# Nome versionado

date_suffix = dt.datetime.now().strftime(
    "%Y-%m-%d"
)

blob_name_with_date = (
    f"{date_suffix}_uf.parquet"
)

export_bigquery_table_to_blob(
    source_table="basedosdados.br_inep_avaliacao_alfabetizacao.uf",
    blob_container=BRONZE_CONTAINER,
    blob_name=blob_name_with_date
)

✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.uf para azure://bronze/2026-08-23_uf.parquet


In [32]:
import datetime as dt

# Lista de tabelas
TABLES = [
    "basedosdados.br_inep_avaliacao_alfabetizacao.uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio"
    #,
   # "basedosdados.br_inep_avaliacao_alfabetizacao.municipio",
   # "basedosdados.br_inep_avaliacao_alfabetizacao.alunos"
]

# Data atual para sufixo
date_suffix = dt.datetime.now().strftime("%Y-%m-%d")
container = os.getenv("AZURE_CONTAINER_BRONZE", "bronze")

# Loop sobre todas as tabelas
for table in TABLES:
# Usa apenas o último pedaço do nome da tabela para o arquivo
    table_suffix = table.split(".")[-1]
    blob_name_with_date = f"{date_suffix}_{table_suffix}.parquet"

    export_bigquery_table_to_blob(
        source_table=table,
        blob_container=container,
        blob_name=blob_name_with_date
    )


✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.uf para azure://bronze/2026-08-23_uf.parquet
✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil para azure://bronze/2026-08-23_meta_alfabetizacao_brasil.parquet
✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf para azure://bronze/2026-08-23_meta_alfabetizacao_uf.parquet
✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio para azure://bronze/2026-08-23_meta_alfabetizacao_municipio.parquet
